# Lab 02 — Advanced HTML & Native Interactivity

**HTML Lesson 2: Beyond the Basics**

**Goal:** Build an interactive student portal with HTML only. Do not add CSS or JavaScript to the page.

## How to work in this notebook

1. Use **File → Save a copy in Drive** and work in your copy.
2. Run the setup cell once. If a later cell says `preview_html` is not defined, run the setup cell again.
3. Run the example and click the preview.
4. Edit the exercise HTML, run that cell again, and read the checks.
5. The checks read your tags and attributes. Click the preview to see what the browser does.

The preview is a page inside a frame. Popovers and dialogs open inside that frame. A form that passes validation stays on screen in the preview. The HTML file that is saved does not include the preview frame.

Download `student-portal.html` from Colab's Files panel and open it in a current browser before you submit. Dialog buttons use `command` and `commandfor`. That works in Chrome 135+, Edge 135+, Firefox 144+, and Safari 26.2+.

**Topics:** attributes, advanced inputs, fieldsets, datalist, details and summary, popovers, dialogs, progress, and meter.


## 1. Setup

Python only saves the page and shows the preview. The page you write is HTML.

You do not need to edit the setup cell. The frame uses the same preview idea as Lab 01: your document is placed in its own page so it cannot change the notebook.

`allow-forms` lets the browser show validation when you submit. Popovers and dialogs are HTML features, so the page does not need a script.


In [ ]:
# Run this cell once. You do not need to edit it.
# preview_html saves your page and shows it in a frame.
# check reads tags and attributes. It does not click the page.
from html import escape
from html.parser import HTMLParser
from IPython.display import HTML, display
from pathlib import Path
import re

VOID_TAGS = {
    "area", "base", "br", "col", "embed", "hr", "img", "input",
    "link", "meta", "source", "track", "wbr",
}


class Element:
    def __init__(self, tag, attrs):
        self.tag = tag
        self.attrs = {}
        for key, value in attrs:
            self.attrs[key.lower()] = "" if value is None else value
        self.children = []
        self.data = ""

    def get(self, name, default=None):
        return self.attrs.get(name.lower(), default)

    def text(self):
        parts = [self.data]
        for child in self.children:
            parts.append(child.text())
        return "".join(parts)

    def find_all(self, tag):
        tag = tag.lower()
        found = [self] if self.tag == tag else []
        for child in self.children:
            found.extend(child.find_all(tag))
        return found

    def find(self, tag):
        found = self.find_all(tag)
        return found[0] if found else None


class DocumentBuilder(HTMLParser):
    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.root = Element("document", [])
        self.stack = [self.root]

    def handle_starttag(self, tag, attrs):
        self._start(tag, attrs)

    def handle_startendtag(self, tag, attrs):
        self._start(tag, attrs, void=True)

    def _start(self, tag, attrs, void=False):
        element = Element(tag.lower(), attrs)
        self.stack[-1].children.append(element)
        if not void and element.tag not in VOID_TAGS:
            self.stack.append(element)

    def handle_endtag(self, tag):
        tag = tag.lower()
        for index in range(len(self.stack) - 1, 0, -1):
            if self.stack[index].tag == tag:
                del self.stack[index:]
                return

    def handle_data(self, data):
        self.stack[-1].data += data


def parse_html(source):
    builder = DocumentBuilder()
    builder.feed(source)
    builder.close()
    return builder.root


def _walk(root):
    yield root
    for child in root.children:
        yield from _walk(child)


def _contains(ancestor, node):
    for child in ancestor.children:
        if child is node or _contains(child, node):
            return True
    return False


def html_only(root):
    for element in _walk(root):
        if element.tag in {"script", "style"}:
            return False
        if element.tag == "link" and "stylesheet" in (element.get("rel") or "").lower():
            return False
        for name in element.attrs:
            if name == "style" or name.startswith("on"):
                return False
    return True


def has_doctype(source):
    return source.lstrip().lower().startswith("<!doctype html>")


def has_lang(root):
    html = root.find("html")
    return html is not None and (html.get("lang") or "").lower() == "en"


def has_title(root):
    title = root.find("title")
    return title is not None and bool(title.text().strip())


def has_charset(root):
    for meta in root.find_all("meta"):
        if (meta.get("charset") or "").lower() == "utf-8":
            return True
        if (meta.get("http-equiv") or "").lower() == "content-type" and "utf-8" in (meta.get("content") or "").lower():
            return True
    return False


def ids_ok(root):
    seen = set()
    for element in _walk(root):
        if "id" not in element.attrs:
            continue
        element_id = (element.get("id") or "").strip()
        if not re.fullmatch(r"[A-Za-z][A-Za-z0-9_-]+", element_id):
            return False
        if element_id in seen:
            return False
        seen.add(element_id)
    return True


def h1_is(root, text):
    heading = root.find("h1")
    return heading is not None and heading.text().strip() == text


def h1_has_title(root):
    heading = root.find("h1")
    return heading is not None and bool((heading.get("title") or "").strip())


def is_editable(element):
    if "contenteditable" not in element.attrs:
        return False
    return (element.get("contenteditable") or "").lower() in {"", "true", "plaintext-only"}


def has_bio(root):
    return any(
        element.tag == "p" and is_editable(element) and element.text().strip()
        for element in _walk(root)
    )


def label_of(root, control):
    control_id = (control.get("id") or "").strip()
    if control_id:
        for label in root.find_all("label"):
            if label.get("for") == control_id and label.text().strip():
                return label.text().strip()
    for label in root.find_all("label"):
        if _contains(label, control) and label.text().strip():
            return label.text().strip()
    return ""


def connected_label(root, control):
    return bool(label_of(root, control))


def in_form(root, element):
    form = root.find("form")
    return element is not None and form is not None and _contains(form, element)


def inputs_of_type(root, type_name):
    wanted = type_name.lower()
    found = []
    for element in root.find_all("input"):
        actual = (element.get("type") or "text").lower()
        if actual == wanted:
            found.append(element)
    return found


def named_labeled_input(root, type_name, required=False, label_word=None):
    for element in inputs_of_type(root, type_name):
        if not in_form(root, element) or not connected_label(root, element):
            continue
        if not (element.get("name") or "").strip():
            continue
        if required and "required" not in element.attrs:
            continue
        if label_word and label_word not in label_of(root, element).lower():
            continue
        return element
    return None


def has_submit(root):
    form = root.find("form")
    if form is None:
        return False
    for button in form.find_all("button"):
        if (button.get("type") or "submit").lower() == "submit" and button.text().strip():
            return True
    return any(True for _ in form.find_all("input") if (_.get("type") or "").lower() == "submit")


def skill_range(root):
    for element in inputs_of_type(root, "range"):
        if not in_form(root, element) or not connected_label(root, element):
            continue
        if not (element.get("name") or "").strip():
            continue
        if _number_is(element, "min", 0) and _number_is(element, "max", 100):
            return element
    return None


def ranged_input(root):
    for element in inputs_of_type(root, "range"):
        if not in_form(root, element) or not connected_label(root, element):
            continue
        if not (element.get("name") or "").strip():
            continue
        if _float_attr(element, "min") is None or _float_attr(element, "max") is None:
            continue
        return element
    return None


def _float_attr(element, name):
    raw = element.get(name)
    if raw is None or str(raw).strip() == "":
        return None
    try:
        return float(raw)
    except (TypeError, ValueError):
        return None


def _number_is(element, name, expected):
    value = _float_attr(element, name)
    return value is not None and value == float(expected)


def has_legend(root, text):
    wanted = text.strip().lower()
    for group in root.find_all("fieldset"):
        elements = [child for child in group.children if child.tag]
        if not elements or elements[0].tag != "legend":
            continue
        if elements[0].text().strip().lower() == wanted:
            return True
    return False


def fieldsets_ready(root):
    form = root.find("form")
    if form is None:
        return False
    groups = form.find_all("fieldset")
    if len(groups) < 2:
        return False
    for group in groups:
        elements = [child for child in group.children if child.tag]
        if not elements or elements[0].tag != "legend" or not elements[0].text().strip():
            return False
    return True


def option_values(element):
    values = []
    for option in element.find_all("option"):
        value = (option.get("value") or option.text()).strip()
        if value:
            values.append(value)
    return values


def datalist_ready(root, minimum):
    for data_list in root.find_all("datalist"):
        list_id = (data_list.get("id") or "").strip()
        if not list_id or len(option_values(data_list)) < minimum:
            continue
        for element in root.find_all("input"):
            if element.get("list") == list_id and in_form(root, element) and connected_label(root, element):
                return True
    return False


def explains_datalist(root):
    for paragraph in root.find_all("p"):
        text = " ".join(paragraph.text().split()).lower()
        if len(text) >= 40 and "datalist" in text and "select" in text:
            return True
    return False


def details_ready(root, minimum):
    complete = 0
    opened = False
    for details in root.find_all("details"):
        elements = [child for child in details.children if child.tag]
        summary = elements[0] if elements and elements[0].tag == "summary" else None
        if summary is None or not summary.text().strip():
            return False
        other = [details.data]
        for child in details.children:
            if child is not summary:
                other.append(child.text())
        if not "".join(other).strip():
            return False
        complete += 1
        if "open" in details.attrs:
            opened = True
    return complete >= minimum and opened


def wired_popovers(root):
    count = 0
    for element in _walk(root):
        if "popover" not in element.attrs:
            continue
        if (element.get("popover") or "").lower() in {"false", "none"}:
            continue
        panel_id = (element.get("id") or "").strip()
        if not panel_id:
            continue
        for button in root.find_all("button"):
            if (button.get("type") or "").lower() != "button":
                continue
            if button.get("popovertarget") == panel_id:
                count += 1
                break
    return count


def dialog_ready(root):
    for dialog in root.find_all("dialog"):
        dialog_id = (dialog.get("id") or "").strip()
        if not dialog_id:
            continue
        opened = False
        closed = False
        for button in root.find_all("button"):
            if (button.get("type") or "").lower() != "button":
                continue
            if button.get("commandfor") != dialog_id:
                continue
            command = (button.get("command") or "").lower()
            opened = opened or command == "show-modal"
            closed = closed or command == "close"
        if opened and closed:
            return True
    return False


def element_by_label(root, tag, word):
    for element in root.find_all(tag):
        if word in label_of(root, element).lower():
            return element
    return None


def percent_full(element):
    if element is None:
        return None
    value = _float_attr(element, "value")
    if value is None:
        return None
    minimum = _float_attr(element, "min")
    maximum = _float_attr(element, "max")
    if minimum is None:
        minimum = 0.0
    if maximum is None:
        maximum = 1.0
    if maximum == minimum:
        return None
    return (value - minimum) / (maximum - minimum)


def near_percent(element, expected):
    ratio = percent_full(element)
    return ratio is not None and abs(ratio - expected) <= 0.011


def meter_scale(element):
    return element is not None and all(_float_attr(element, name) is not None for name in ("low", "high", "optimum"))


def has_value_and_max(element):
    return element is not None and _float_attr(element, "value") is not None and _float_attr(element, "max") is not None


_FORM_TAG = re.compile(r"<form\b[^>]*>", re.IGNORECASE)
_DIALOG_METHOD = re.compile(r"""\bmethod\s*=\s*(?:"dialog"|'dialog'|dialog\b)""", re.IGNORECASE)


def _rewrite_form_tag(tag):
    if _DIALOG_METHOD.search(tag):
        return tag
    extras = []
    if not re.search(r"\btarget\s*=", tag, re.IGNORECASE):
        extras.append('target="lab02-preview-sink"')
    if not re.search(r"\baction\s*=", tag, re.IGNORECASE):
        extras.append('action="about:blank"')
    if not extras:
        return tag
    return tag[:-1] + " " + " ".join(extras) + ">"


def preview_document(source):
    """Preview copy only. The saved file stays exactly as you wrote it."""
    updated = _FORM_TAG.sub(lambda match: _rewrite_form_tag(match.group(0)), source)
    meta = '<meta name="color-scheme" content="light">'
    sink = '<iframe name="lab02-preview-sink" hidden></iframe>'
    lower = updated.lower()
    head_at = lower.find("<head")
    if head_at != -1:
        head_end = updated.find(">", head_at)
        if head_end != -1:
            updated = updated[: head_end + 1] + meta + updated[head_end + 1 :]
    else:
        updated = meta + updated
    lower = updated.lower()
    body_at = lower.find("<body")
    if body_at != -1:
        body_end = updated.find(">", body_at)
        if body_end != -1:
            updated = updated[: body_end + 1] + sink + updated[body_end + 1 :]
    else:
        updated = sink + updated
    return updated


def preview_html(source, filename="preview.html", height=480):
    if not isinstance(source, str):
        raise TypeError("Pass your HTML string, for example preview_html(exercise_a, 'exercise_a.html').")
    if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]*\.html", filename):
        raise ValueError("Use a simple file name ending in .html.")
    Path(filename).write_text(source, encoding="utf-8")
    framed = preview_document(source)
    safe_name = escape(filename)
    height_px = max(240, min(int(height), 1200))
    # The sandbox lets the frame validate forms and open dialogs without
    # giving the page access to the notebook.
    markup = (
        f'<p style="margin:0 0 8px;font:14px/1.4 sans-serif">'
        f"Preview of <code>{safe_name}</code>. Click inside the frame to use the controls."
        f"</p>"
        f'<iframe title="Preview of {safe_name}" '
        f'sandbox="allow-forms allow-modals allow-popups" '
        f'style="width:100%;height:{height_px}px;border:1px solid #aaa;border-radius:8px;'
        f'background:#fff;color-scheme:light" '
        f'srcdoc="{escape(framed, quote=True)}"></iframe>'
    )
    display(HTML(markup))
    print(f"Saved {filename}. In Colab, download it from the Files panel and open it in Chrome.")


def check(source, tests):
    root = parse_html(source)
    passed = 0
    for label, test in tests:
        try:
            ok = bool(test(root, source))
        except Exception:
            ok = False
        print(("✅ PASS" if ok else "❌ FIX ") + " — " + label)
        passed += int(ok)
    print(f"\nResult: {passed}/{len(tests)} checks passed")
    print("These checks read your HTML. They do not click the preview.")
    return passed == len(tests)


print("Ready. Run an example, use the preview, then edit the exercise below it.")


## 2. Exercise A — Attributes and advanced inputs

**Objective:** Use attributes and input types that the browser understands without JavaScript.

**Explanation**

- `title` on an element shows a tooltip when the pointer rests on it.
- `contenteditable="true"` lets someone change that text in the browser.
- `type` changes the control: `date`, `color`, and `range`.
- `required` blocks submit until the field has a value. The submit button is what starts that check.
- A label's `for` value must match the control's `id`. Give the control a `name` as well.

**Expected behavior**

- Rest on the heading and a tooltip appears.
- Click the bio and type.
- The date, color, and range controls open their own browser UI.
- Submit with an empty name and the browser asks you to fill it in. Submit with a name and the preview stays on the page.

**Your turn**

In the exercise cell, keep the heading text **Student Details**. Add a `title` tooltip, an editable bio, and labeled form fields for name (required text), birthday (`date`), favorite color (`color`), and skill level (`range` from 0 to 100). Keep the submit button.

**Additional challenge:** Add `placeholder` and `autocomplete="name"` on the name field. The checks do not grade the challenge.


In [ ]:
# Example. Run this cell and click the preview. Then edit the exercise cell.
demo_a = """<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <title>Attribute example</title>
</head>
<body>
  <h1 title="This tooltip comes from the title attribute">Hover this heading</h1>
  <p contenteditable="true">Click this bio and type a new sentence.</p>
  <form>
    <label for="practice-name">Name:</label>
    <input id="practice-name" name="practice-name" type="text" required>
    <br>
    <label for="practice-date">Sample date:</label>
    <input id="practice-date" name="practice-date" type="date">
    <br>
    <label for="practice-color">Sample color:</label>
    <input id="practice-color" name="practice-color" type="color">
    <br>
    <label for="practice-level">Practice level:</label>
    <input id="practice-level" name="practice-level" type="range" min="0" max="100" value="40">
    <br>
    <button type="submit">Try submit</button>
  </form>
</body>
</html>"""

preview_html(demo_a, "demo_a.html", height=460)
check(demo_a, [
    ("Example is HTML only", lambda root, source: html_only(root)),
    ("Heading has a title tooltip", lambda root, source: h1_has_title(root)),
    ("Example has editable text", lambda root, source: has_bio(root)),
    ("Example has a required name field", lambda root, source: named_labeled_input(root, "text", required=True, label_word="name") is not None),
    ("Example has date, color, and range fields", lambda root, source: all([
        named_labeled_input(root, "date"),
        named_labeled_input(root, "color"),
        skill_range(root),
    ])),
    ("Example form has a submit button", lambda root, source: has_submit(root)),
])


In [ ]:
# Your turn. Edit this HTML, run the cell again, and use the preview.
exercise_a = """<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <title>Student Details</title>
</head>
<body>
  <h1>Student Details</h1>
  <!-- TODO: Add a title attribute to the heading so a tooltip appears. -->
  <!-- TODO: Add a paragraph with contenteditable="true" for a short bio. -->
  <form>
    <!-- TODO: Add labeled inputs with name and id:
         name (required text), birthday (date), favorite color (color),
         and skill level (range, min 0, max 100). -->
    <button type="submit">Save details</button>
  </form>
</body>
</html>"""

preview_html(exercise_a, "exercise_a.html", height=480)
check(exercise_a, [
    ("Page starts with an HTML5 doctype", lambda root, source: has_doctype(source)),
    ("Page language is English", lambda root, source: has_lang(root)),
    ("Title is Student Details", lambda root, source: has_title(root) and root.find("title").text().strip() == "Student Details"),
    ("Page is HTML only, with no CSS or JavaScript", lambda root, source: html_only(root)),
    ("Heading text is Student Details", lambda root, source: h1_is(root, "Student Details")),
    ("Heading has a non-empty title tooltip", lambda root, source: h1_has_title(root)),
    ("Bio can be edited and already contains text", lambda root, source: has_bio(root)),
    ("Required name field has a label that includes the word name", lambda root, source: named_labeled_input(root, "text", required=True, label_word="name") is not None),
    ("Form has a labeled date field", lambda root, source: named_labeled_input(root, "date") is not None),
    ("Form has a labeled color field", lambda root, source: named_labeled_input(root, "color") is not None),
    ("Skill level is a range from 0 to 100 with a label", lambda root, source: skill_range(root) is not None),
    ("Form has a submit button", lambda root, source: has_submit(root)),
])


## 3. Exercise B — Fieldset, legend, and datalist

**Objective:** Group controls, and offer suggested values without locking the field to a list.

**Explanation**

- `fieldset` groups related controls. `legend` is the group's name. The legend is the first element inside the fieldset.
- `datalist` suggests values for a text field. The input uses `list="the datalist id"`. The person can still type a value that is not in the list.
- `select` allows only its `option` elements. A datalist does not work that way.

**Expected behavior**

- The example shows one group, a course box with suggestions, and a select with the same two courses.
- In your page, Course Preferences is its own group. The course box suggests at least four names and still accepts a course you type yourself.
- Your paragraph uses the words datalist and select and says how they differ.

**Your turn**

Keep the Personal Details group and its email field. Add a fieldset whose legend is **Course Preferences**. Add a labeled course input and a datalist with at least four suggestions. HTML and CSS can be two of them. Add at least two more. Then add a paragraph, in your own words, that uses the words datalist and select.

**Additional challenge:** Add a `select` with the same courses so you can compare the two controls. The checks do not grade the challenge.


In [ ]:
# Example. Run this cell and try both course controls.
demo_b = """<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <title>Fieldset example</title>
</head>
<body>
  <h1>Course controls</h1>
  <form>
    <fieldset>
      <legend>Contact</legend>
      <label for="demo-email">Email:</label>
      <input id="demo-email" name="demo-email" type="email" required>
    </fieldset>
    <p>
      <label for="demo-course">Suggested course:</label>
      <input id="demo-course" name="demo-course" list="demo-courses">
      <datalist id="demo-courses">
        <option value="HTML">
        <option value="CSS">
      </datalist>
    </p>
    <p>
      <label for="demo-select">Listed course:</label>
      <select id="demo-select" name="demo-select">
        <option value="HTML">HTML</option>
        <option value="CSS">CSS</option>
      </select>
    </p>
    <button type="submit">Try submit</button>
  </form>
  <p>A datalist suggests values and still lets you type a course that is not on the list. A select allows only the options written inside it.</p>
</body>
</html>"""

preview_html(demo_b, "demo_b.html", height=480)
check(demo_b, [
    ("Example is HTML only", lambda root, source: html_only(root)),
    ("Example groups the email field with a legend", lambda root, source: has_legend(root, "Contact")),
    ("Example email field is required", lambda root, source: named_labeled_input(root, "email", required=True, label_word="email") is not None),
    ("Example datalist is connected to an input", lambda root, source: datalist_ready(root, 2)),
    ("Example has a select", lambda root, source: root.find("select") is not None and connected_label(root, root.find("select"))),
    ("Example explains datalist and select", lambda root, source: explains_datalist(root)),
])


In [ ]:
# Your turn. Edit this HTML, run the cell again, and use the preview.
exercise_b = """<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <title>Course Registration</title>
</head>
<body>
  <h1>Course Registration</h1>
  <form>
    <fieldset>
      <legend>Personal Details</legend>
      <label for="email">Email:</label>
      <input id="email" name="email" type="email" required>
    </fieldset>
    <!-- TODO: Add a fieldset with the legend Course Preferences.
         Add a labeled course input and a datalist with at least four
         suggestions. Connect the input with the list attribute. -->
    <!-- TODO: Add a paragraph that uses the words datalist and select
         and explains the difference in your own words. -->
    <button type="submit">Register</button>
  </form>
</body>
</html>"""

preview_html(exercise_b, "exercise_b.html", height=520)
check(exercise_b, [
    ("Page starts with an HTML5 doctype", lambda root, source: has_doctype(source)),
    ("Page language is English", lambda root, source: has_lang(root)),
    ("Page is HTML only, with no CSS or JavaScript", lambda root, source: html_only(root)),
    ("Personal Details group is present", lambda root, source: has_legend(root, "Personal Details")),
    ("Email field is required and its label includes the word email", lambda root, source: named_labeled_input(root, "email", required=True, label_word="email") is not None),
    ("Course Preferences group is present", lambda root, source: has_legend(root, "Course Preferences")),
    ("Course field uses a datalist with at least four suggestions", lambda root, source: datalist_ready(root, 4)),
    ("A paragraph explains datalist and select", lambda root, source: explains_datalist(root)),
    ("Form has a submit button", lambda root, source: has_submit(root)),
])


## 4. Exercise C — Native interactive HTML

**Objective:** Open and close parts of a page with HTML, without writing JavaScript.

**Explanation**

- `details` and `summary` are a disclosure. The summary is the visible control. `open` shows the answer when the page loads. Click the summary again to close it. With the summary focused, Enter or Space toggles it.
- `popover` on an element makes a small panel. A `button type="button"` with `popovertarget="the panel id"` opens and closes it. Use `type="button"` so the button does not submit a form.
- `dialog` is a window. `commandfor="the dialog id"` and `command="show-modal"` open it as a modal, so the rest of the page does not take clicks. `command="close"` on another button closes it. A current browser also closes a modal dialog with Escape.
- These dialog buttons need Chrome 135+, Edge 135+, Firefox 144+, or Safari 26.2+. The preview draws the panel and the dialog inside the frame.

**Expected behavior**

- The example FAQ opens and closes.
- Show Help opens the panel. The button inside the panel closes it.
- Open Information opens a dialog that blocks the page behind it. Close dismisses it.

**Your turn**

Keep the first question and the first help panel. Add two more questions (three `details` in total, one of them `open`). Add a second popover with its own button. Add a dialog with an Open button (`command="show-modal"`) and a Close button (`command="close"`). Every one of those buttons needs `type="button"`.

**Additional challenge:** Add a question whose summary is "How do I close a dialog?" The checks do not grade the challenge.


In [ ]:
# Example. Run this cell. Open the question, the help panel, and the dialog.
demo_c = """<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <title>Help example</title>
</head>
<body>
  <h1>Help example</h1>
  <details open>
    <summary>What is HTML?</summary>
    <p>HTML describes the structure of a webpage.</p>
  </details>
  <details>
    <summary>Do I need JavaScript for this question?</summary>
    <p>No. The browser opens and closes the answer.</p>
  </details>
  <p>
    <button type="button" popovertarget="demo-help">Show Help</button>
  </p>
  <div id="demo-help" popover>
    <p>Ask your instructor if you get stuck.</p>
    <button type="button" popovertarget="demo-help">Hide Help</button>
  </div>
  <p>
    <button type="button" commandfor="demo-dialog" command="show-modal">Open Information</button>
  </p>
  <dialog id="demo-dialog">
    <h2>Information</h2>
    <p>Keep practicing HTML.</p>
    <button type="button" commandfor="demo-dialog" command="close">Close</button>
  </dialog>
</body>
</html>"""

preview_html(demo_c, "demo_c.html", height=520)
check(demo_c, [
    ("Example is HTML only", lambda root, source: html_only(root)),
    ("Example has two complete questions and one is open", lambda root, source: details_ready(root, 2)),
    ("Example help button opens a popover", lambda root, source: wired_popovers(root) >= 1),
    ("Example dialog has Open and Close buttons", lambda root, source: dialog_ready(root)),
])


In [ ]:
# Your turn. Edit this HTML, run the cell again, and use the preview.
exercise_c = """<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <title>Help Center</title>
</head>
<body>
  <h1>Student Help Center</h1>
  <details open>
    <summary>What is HTML?</summary>
    <p>HTML describes the structure of a webpage.</p>
  </details>
  <!-- TODO: Add two more details elements. Each needs a summary and an answer. -->

  <p>
    <button type="button" popovertarget="help">Show Help</button>
  </p>
  <div id="help" popover>
    <p>Ask your instructor if you get stuck.</p>
  </div>
  <!-- TODO: Add a second popover with its own id, and a type="button"
       whose popovertarget matches that id. -->

  <!-- TODO: Add a dialog with an id.
       Open button: type="button" command="show-modal" commandfor="that id".
       Close button: type="button" command="close" commandfor="that id". -->
</body>
</html>"""

preview_html(exercise_c, "exercise_c.html", height=560)
check(exercise_c, [
    ("Page starts with an HTML5 doctype", lambda root, source: has_doctype(source)),
    ("Page language is English", lambda root, source: has_lang(root)),
    ("Page is HTML only, with no CSS or JavaScript", lambda root, source: html_only(root)),
    ("Three questions each have a summary and an answer, and one is open", lambda root, source: details_ready(root, 3)),
    ("Two popovers each have a type=button control", lambda root, source: wired_popovers(root) >= 2),
    ("A dialog has show-modal and close buttons", lambda root, source: dialog_ready(root)),
])


## 5. Exercise D — Progress and meter

**Objective:** Choose the element that matches the kind of number you are showing.

**Explanation**

- `progress` is how far a task has gone, such as course completion. Set `value` and `max`.
- `meter` is a measurement on a known scale, such as storage used. `min` and `max` are the scale. `low`, `high`, and `optimum` tell the browser which part of the scale is a warning and which part is the goal. When storage sits above `high`, the meter can show a warning color.
- A label's `for` value must match the element's `id`.

**Expected behavior**

- The example shows a task bar and a storage gauge.
- After your edit, course completion is 45% of its max, and storage used is 90% of its scale.

**Your turn**

Change the dashboard. Set course completion to 45% (`value="45"` and `max="100"`). Set storage to 90%. The shortest edit is `value="0.9"` on the meter, leaving `max="1"`. You can use `value="90"` and `max="100"` instead. If you change `max`, keep `low`, `high`, and `optimum` on that same scale.

In your own words, decide when you would use each element. You can write that in a paragraph. The checks look at the numbers, not at that sentence.

**Additional challenge:** Add a second meter for a quiz score, with `low`, `high`, and `optimum`. The checks do not grade the challenge.


In [ ]:
# Example. Run this cell and compare the two gauges.
demo_d = """<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <title>Gauge example</title>
</head>
<body>
  <h1>Progress and meter</h1>
  <p>Progress shows how much of a task is finished. Meter shows a measured amount on a scale.</p>
  <p>
    <label for="demo-progress">Lesson completion:</label>
    <progress id="demo-progress" value="70" max="100">70%</progress>
  </p>
  <p>
    <label for="demo-storage">Drive space:</label>
    <meter id="demo-storage" value="0.8" min="0" max="1" low="0.25" high="0.75" optimum="0.2">80%</meter>
  </p>
</body>
</html>"""

preview_html(demo_d, "demo_d.html", height=320)
check(demo_d, [
    ("Example is HTML only", lambda root, source: html_only(root)),
    ("Example progress has a label, value, and max", lambda root, source: has_value_and_max(element_by_label(root, "progress", "completion"))),
    ("Example meter has a label, value, max, low, high, and optimum", lambda root, source: has_value_and_max(element_by_label(root, "meter", "space")) and meter_scale(element_by_label(root, "meter", "space"))),
])


In [ ]:
# Your turn. Change the two values, run the cell again, and look at the preview.
exercise_d = """<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <title>Student Dashboard</title>
</head>
<body>
  <h1>Student Dashboard</h1>
  <p>
    <label for="progress">Course completion:</label>
    <progress id="progress" value="70" max="100">70%</progress>
  </p>
  <p>
    <label for="storage">Storage used:</label>
    <meter id="storage" value="0.8" min="0" max="1" low="0.25" high="0.75" optimum="0.2">80%</meter>
  </p>
</body>
</html>"""

preview_html(exercise_d, "exercise_d.html", height=320)
check(exercise_d, [
    ("Page starts with an HTML5 doctype", lambda root, source: has_doctype(source)),
    ("Page is HTML only, with no CSS or JavaScript", lambda root, source: html_only(root)),
    ("Course completion is 45 percent", lambda root, source: near_percent(element_by_label(root, "progress", "completion"), 0.45)),
    ("Storage used is 90 percent of its scale", lambda root, source: near_percent(element_by_label(root, "meter", "storage"), 0.90)),
    ("Storage meter still has low, high, and optimum", lambda root, source: meter_scale(element_by_label(root, "meter", "storage"))),
])


## 6. Final challenge — Interactive student portal

**Objective:** Combine the controls from this lab in one HTML page.

Create `student-portal.html` with HTML only. The checks below match this list.

1. An HTML5 doctype, `lang="en"`, a UTF-8 charset, and a non-empty title.
2. One form with two fieldsets. Each fieldset has a legend.
3. Labeled controls, each with `name` and `id`, for a required name, an email, a date, a color, and a range that has `min` and `max`. Include a submit button.
4. A datalist with at least two course suggestions, connected with `list`.
5. Three `details` questions. Each has a `summary` and an answer. One has `open`.
6. A help popover opened by a `button type="button"` and `popovertarget`.
7. A `dialog` with a `type="button"` Open control (`command="show-modal"`) and a Close control (`command="close"`).
8. A labeled `progress` and a labeled `meter`. Each has `value` and `max`.
9. A paragraph with `contenteditable="true"` that already contains text.
10. Unique ids that start with a letter and are at least two characters. No CSS and no JavaScript.

**Expected behavior**

The preview shows the portal inside the frame. Questions open, the help panel opens, the dialog blocks the rest of the preview, and an empty required name stops submit.

**Additional challenge:** Add a file input, a second popover, and `autocomplete="email"` on the email field. The checks do not grade the challenge.

**Submission:** Download `student-portal.html`, open it in Chrome or another current browser, and submit that file. The saved file does not contain the preview frame.


In [ ]:
# Your turn. Replace the TODO comments with your portal, then run this cell again.
student_portal = """<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Interactive Student Portal</title>
</head>
<body>
  <header>
    <h1>Interactive Student Portal</h1>
  </header>
  <main>
    <!-- TODO 1: Add one form with two fieldsets and legends.
         Add labeled inputs with name and id:
         required name, email, date, color, and range with min and max.
         Add a submit button. -->
    <!-- TODO 2: Add a datalist of courses and a labeled input that uses list. -->
    <!-- TODO 3: Add a progress element and a meter.
         Give each a label, a value, and a max. -->
    <!-- TODO 4: Add three details questions. One should include open. -->
    <!-- TODO 5: Add a help popover and a type="button" that uses popovertarget. -->
    <!-- TODO 6: Add a dialog.
         Open: type="button" command="show-modal" commandfor="the dialog id".
         Close: type="button" command="close" commandfor="the same id". -->
    <!-- TODO 7: Add a short bio with contenteditable="true". -->
  </main>
  <footer>
    <p>HTML Lab 02</p>
  </footer>
</body>
</html>"""

preview_html(student_portal, "student-portal.html", height=640)
check(student_portal, [
    ("Page starts with an HTML5 doctype", lambda root, source: has_doctype(source)),
    ("Page language is English", lambda root, source: has_lang(root)),
    ("Charset is UTF-8", lambda root, source: has_charset(root)),
    ("Title is not empty", lambda root, source: has_title(root)),
    ("Page is HTML only, with no CSS or JavaScript", lambda root, source: html_only(root)),
    ("Every id is unique, starts with a letter, and has at least two characters", lambda root, source: ids_ok(root)),
    ("The form has two fieldsets and each has a legend", lambda root, source: fieldsets_ready(root)),
    ("Required name field has a label that includes the word name", lambda root, source: named_labeled_input(root, "text", required=True, label_word="name") is not None),
    ("Email field has a label that includes the word email", lambda root, source: named_labeled_input(root, "email", label_word="email") is not None),
    ("Date is a labeled date field", lambda root, source: named_labeled_input(root, "date") is not None),
    ("Color is a labeled color field", lambda root, source: named_labeled_input(root, "color") is not None),
    ("Range is labeled and has min and max", lambda root, source: ranged_input(root) is not None),
    ("The form has a submit button", lambda root, source: has_submit(root)),
    ("A labeled input uses a datalist with at least two suggestions", lambda root, source: datalist_ready(root, 2)),
    ("Three questions each have a summary and an answer, and one is open", lambda root, source: details_ready(root, 3)),
    ("A type=button control opens a popover", lambda root, source: wired_popovers(root) >= 1),
    ("A dialog has show-modal and close buttons", lambda root, source: dialog_ready(root)),
    ("Progress has a label, value, and max", lambda root, source: any(has_value_and_max(element) and connected_label(root, element) for element in root.find_all("progress"))),
    ("Meter has a label, value, and max", lambda root, source: any(has_value_and_max(element) and connected_label(root, element) for element in root.find_all("meter"))),
    ("Bio can be edited and already contains text", lambda root, source: has_bio(root)),
])


## 7. Self-check and reflection

Run every exercise cell. A passing check means the tags and attributes are in place. It does not mean the checker clicked the page. Use the preview for that.

- Does every form control have a label whose `for` matches the control `id`?
- Can you type a course that is not in the datalist?
- Does each question open and close, including with the keyboard on the summary?
- Does the help panel open and close inside the preview frame?
- Does the dialog block the preview behind it, and does Close dismiss it?
- Is course completion a `progress` element, and is the measured amount a `meter`?
- Is the saved page free of CSS and JavaScript?

**Reflection:** Which native HTML feature surprised you most? When would you still need JavaScript?

The preview frame is there so Colab can show a full document. Popovers and dialogs are drawn inside that frame, not over the Colab menus. Open the downloaded file in a current browser when you want to see the page on its own.

### Further reading

- [MDN: details](https://developer.mozilla.org/en-US/docs/Web/HTML/Element/details)
- [MDN: datalist](https://developer.mozilla.org/en-US/docs/Web/HTML/Element/datalist)
- [MDN: popover](https://developer.mozilla.org/en-US/docs/Web/HTML/Reference/Global_attributes/popover)
- [MDN: dialog](https://developer.mozilla.org/en-US/docs/Web/HTML/Element/dialog)
- [MDN: Invoker Commands](https://developer.mozilla.org/en-US/docs/Web/API/Invoker_Commands_API)
- [MDN: progress](https://developer.mozilla.org/en-US/docs/Web/HTML/Element/progress)
- [MDN: meter](https://developer.mozilla.org/en-US/docs/Web/HTML/Element/meter)
